# Step 4: Pre-Processing and Training Data Development
## Automated Dog Breed Identification from Shelter Photos

## 1. Imports and Load Data

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

MANIFEST_PATH = "C:\\Users\\alyss\\OneDrive\\Springboard Data Science Bootcamp\\Capstone 3\\data\\manifest.csv"
manifest = pd.read_csv(MANIFEST_PATH)
print(f"Loaded manifest: {len(manifest)} rows")
manifest["source"].value_counts()


Loaded manifest: 28355 rows


source
stanford_dogs    20486
petfinder         7869
Name: count, dtype: int64

## 2. Encode the Target (Breed Label)

This is the image-classification equivalent of "create dummy/indicator features for
categorical variables." `breed_harmonized` is a string category (the response variable, not
a predictor) — a classifier needs it as an integer class index. The label encoder is fit
**only on Stanford Dogs' breed vocabulary**, since that's the fixed set of classes the model
will actually learn to predict; PetFinder's `breed_harmonized` values (already mapped onto
this same vocabulary back in the wrangling step) are encoded against that same fitted
mapping, not re-fit — otherwise train-time and evaluation-time class indices could disagree.


In [2]:
stanford_mask = manifest["source"] == "stanford_dogs"

label_encoder = LabelEncoder()
label_encoder.fit(manifest.loc[stanford_mask, "breed_harmonized"])
NUM_CLASSES = len(label_encoder.classes_)
print(f"{NUM_CLASSES} breed classes")

# PetFinder's "no_match" rows have no valid class to encode -- keep them out of anything
# that needs an encoded label; they simply aren't part of the breed-identification task.
encodable_mask = manifest["breed_harmonized"].isin(label_encoder.classes_)
manifest["label"] = -1
manifest.loc[encodable_mask, "label"] = label_encoder.transform(manifest.loc[encodable_mask, "breed_harmonized"])

print(f"Rows with a valid encoded label: {encodable_mask.sum()} / {len(manifest)}")
print(f"PetFinder rows excluded (no_match, unencodable): {((manifest['source']=='petfinder') & ~encodable_mask).sum()}")


115 breed classes
Rows with a valid encoded label: 22090 / 28355
PetFinder rows excluded (no_match, unencodable): 6265


## 3. Standardize Feature Magnitude (Pixel Normalization)

The image equivalent of scaling numeric features: raw pixel values (0-255 per channel) have
far larger magnitude than a neural net expects, and different color channels can have
different scale properties. Rather than picking normalization stats ourselves, we use the
**same preprocessor the pretrained checkpoint itself was trained with** — this matters for
transfer learning specifically, since a mismatch here quietly degrades how much the
pretrained weights actually help.

In [3]:
from transformers import AutoImageProcessor

CHECKPOINT = "google/mobilenet_v2_1.0_224"   
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)

def preprocess_image(path):
    img = Image.open(path).convert("RGB")
    return processor(img, return_tensors="pt")["pixel_values"][0]

preprocessor_config.json:   0%|          | 0.00/406 [00:00<?, ?B/s]

C:\Users\alyss\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\alyss\.cache\huggingface\hub\models--google--mobilenet_v2_1.0_224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## 4. Train / Validation / Test Split

The primary split is stratified by breed, entirely within Stanford Dogs. A validation split (separate from the final test set) is included because fine-tuning a neural net benefits from monitoring performance during training for model selection/early stopping — the test set stays completely untouched until final evaluation.

- **70% train / 15% validation / 15% test**, stratified by `breed_harmonized`
- Checked against the smallest class's count (from EDA: ~143 images) to confirm every split
  still gets a reasonable number of that breed — 143 × 0.15 ≈ 21, comfortably enough.


In [4]:
stanford_df = manifest[stanford_mask & encodable_mask].reset_index(drop=True)

min_class_count = stanford_df["breed_harmonized"].value_counts().min()
print(f"Smallest Stanford breed class: {min_class_count} images (must be large enough to stratify 3 ways)")

train_df, temp_df = train_test_split(
    stanford_df, test_size=0.30, stratify=stanford_df["label"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42
)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"Every class represented in test set: {test_df['breed_harmonized'].nunique() == NUM_CLASSES}")


Smallest Stanford breed class: 143 images (must be large enough to stratify 3 ways)
Train: 14340  Val: 3073  Test: 3073
Every class represented in test set: True


**Secondary real-world evaluation set:** PetFinder rows with a valid breed match, kept
completely separate from the split above — never used for training or hyperparameter
selection, only for a final, secondary check on real-world generalization.

In [5]:
realworld_eval_df = manifest[(manifest["source"] == "petfinder") & encodable_mask].reset_index(drop=True)
print(f"Real-world evaluation set (PetFinder, matched breeds only): {len(realworld_eval_df)} rows")
print(f"Breeds represented: {realworld_eval_df['breed_harmonized'].nunique()} of {NUM_CLASSES} total classes")


Real-world evaluation set (PetFinder, matched breeds only): 1604 rows
Breeds represented: 61 of 115 total classes


## 5. PyTorch Dataset and DataLoaders

Wraps the encoding + normalization steps above into something a training loop (or the
`transformers.Trainer` used in the modeling notebook) can consume directly.


In [6]:
class BreedDataset(Dataset):
    def __init__(self, df):
        self.paths = df["image_path"].values
        self.labels = df["label"].values

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        pixel_values = preprocess_image(self.paths[idx])
        label = int(self.labels[idx])
        return {"pixel_values": pixel_values, "labels": label}

train_dataset = BreedDataset(train_df)
val_dataset = BreedDataset(val_df)
test_dataset = BreedDataset(test_df)
realworld_dataset = BreedDataset(realworld_eval_df)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
realworld_loader = DataLoader(realworld_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"{len(train_loader)} train batches, {len(val_loader)} val batches, "
      f"{len(test_loader)} test batches, {len(realworld_loader)} real-world eval batches")


449 train batches, 97 val batches, 97 test batches, 51 real-world eval batches


### Sanity check: pull one real batch through the full pipeline

In [7]:
batch = next(iter(train_loader))
print("pixel_values shape:", batch["pixel_values"].shape)   # should be [batch_size, 3, 224, 224]
print("labels shape:", batch["labels"].shape)
print("labels dtype:", batch["labels"].dtype)
print("pixel value range (should be roughly -2 to 2 after normalization):",
      batch["pixel_values"].min().item(), "to", batch["pixel_values"].max().item())


pixel_values shape: torch.Size([32, 3, 224, 224])
labels shape: torch.Size([32])
labels dtype: torch.int64
pixel value range (should be roughly -2 to 2 after normalization): -1.0 to 1.0


## 6. Save Splits for the Modeling Notebook

In [8]:
SPLITS_DIR = r"C:\Users\alyss\OneDrive\Springboard Data Science Bootcamp\Capstone 3\data\splits"
os.makedirs(SPLITS_DIR, exist_ok=True)
train_df.to_csv(f"{SPLITS_DIR}/train.csv", index=False)
val_df.to_csv(f"{SPLITS_DIR}/val.csv", index=False)
test_df.to_csv(f"{SPLITS_DIR}/test.csv", index=False)
realworld_eval_df.to_csv(f"{SPLITS_DIR}/realworld_eval.csv", index=False)

with open(f"{SPLITS_DIR}/label_classes.json", "w") as f:
    json.dump(list(label_encoder.classes_), f)

## 7. Summary

- Breed labels encoded to integer class indices via a label encoder fit on Stanford Dogs'
  120-class vocabulary; PetFinder's already-harmonized labels reuse this same encoding
  rather than being fit separately, so class indices agree across both datasets.
- Pixel normalization deferred to the pretrained checkpoint's own image processor, so
  transfer learning gets input statistics consistent with what the checkpoint was trained on.
- Train/val/test split is stratified and **entirely within Stanford Dogs**, per mentor
  feedback — PetFinder's matched subset is preserved as a separate real-world evaluation
  set rather than folded into the primary split.
- Every split, plus the label encoding order, is saved to disk so the modeling notebook
  loads a fixed, reproducible split rather than re-deriving one on every run.
